In [11]:
import tensorflow as tf
tf.keras.backend.clear_session()

import os
import numpy as np

# ✅ FIX: handle corrupted images
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
from tensorflow.keras import layers, models

In [12]:
BASE_DIR = r"C:/Users/raksh/x-ai for medical imaging/data/bone_xray"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

MODEL_SAVE_PATH = "backend/saved_models/bone_fracture_model.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_PHASE1 = 12
EPOCHS_PHASE2 = 8

In [13]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("Class mapping:", train_gen.class_indices)

Found 1400 images belonging to 2 classes.
Found 300 images belonging to 2 classes.
Found 299 images belonging to 2 classes.
Class mapping: {'FRACTURE': 0, 'NORMAL': 1}


In [14]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,218,788 (16.09 MB)

 Trainable params: 166,657 (651.00 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [16]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "backend/saved_models/bone_phase1.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

In [18]:
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks
)

Epoch 1/12
44/44 ━━━━━━━━━━━━━━━━━━━━ 290s 7s/step - accuracy: 0.9829 - loss: 0.0567 - val_accuracy: 0.9900 - val_loss: 0.0565 - learning_rate: 3.0000e-04
Epoch 2/12
44/44 ━━━━━━━━━━━━━━━━━━━━ 175s 4s/step - accuracy: 0.9821 - loss: 0.0453 - val_accuracy: 0.9833 - val_loss: 0.0547 - learning_rate: 3.0000e-04
Epoch 3/12
44/44 ━━━━━━━━━━━━━━━━━━━━ 121s 3s/step - accuracy: 0.9871 - loss: 0.0415 - val_accuracy: 0.9833 - val_loss: 0.0457 - learning_rate: 9.0000e-05
Epoch 4/12
44/44 ━━━━━━━━━━━━━━━━━━━━ 103s 2s/step - accuracy: 0.9929 - loss: 0.0359 - val_accuracy: 0.9867 - val_loss: 0.0413 - learning_rate: 9.0000e-05
Epoch 5/12
44/44 ━━━━━━━━━━━━━━━━━━━━ 68s 2s/step - accuracy: 0.9886 - loss: 0.0415 - val_accuracy: 0.9867 - val_loss: 0.0399 - learning_rate: 9.0000e-05


In [19]:
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [20]:
callbacks_ft = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

In [21]:
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_ft
)

Epoch 1/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 170s 3s/step - accuracy: 0.8743 - loss: 0.3049 - val_accuracy: 0.9800 - val_loss: 0.0569 - learning_rate: 1.0000e-05
Epoch 2/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 89s 2s/step - accuracy: 0.9021 - loss: 0.2453 - val_accuracy: 0.9800 - val_loss: 0.0651 - learning_rate: 1.0000e-05
Epoch 3/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 112s 3s/step - accuracy: 0.9136 - loss: 0.2239 - val_accuracy: 0.9800 - val_loss: 0.0670 - learning_rate: 1.0000e-05
Epoch 4/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 124s 3s/step - accuracy: 0.9271 - loss: 0.1680 - val_accuracy: 0.9733 - val_loss: 0.0758 - learning_rate: 3.0000e-06
Epoch 5/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 125s 3s/step - accuracy: 0.9414 - loss: 0.1651 - val_accuracy: 0.9700 - val_loss: 0.0825 - learning_rate: 3.0000e-06


In [ ]:
test_loss, test_acc = model.evaluate(test_gen)
print("✅ Bone Model Test Accuracy:", test_acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.9766 - loss: 0.0742
✅ Bone Model Test Accuracy: 0.9765886068344116


In [23]:
model = tf.keras.models.load_model("backend/saved_models/bone_phase1.keras")

In [28]:
test_loss, test_acc = model.evaluate(test_gen)
print("✅ Final Test Accuracy:", test_acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.9833 - loss: 0.0628
✅ Final Test Accuracy: 0.9832776188850403


In [25]:
model.save("backend/saved_models/bone_final.keras")
print("✅ Final model saved")

✅ Final model saved


In [27]:
x_batch, y_batch = next(test_gen)

preds = model.predict(x_batch)

pred_labels = (preds > 0.5).astype(int).flatten()
true_labels = y_batch.astype(int)

print("Predicted:", pred_labels[:10])
print("True     :", true_labels[:10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted: [0 0 0 0 0 0 0 0 0 0]
True     : [0 0 0 0 0 0 0 0 0 0]
